In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import pairwise_distances

# Extracting embeddings from a model


## sparse_128_L32_b4_cw6

In [ ]:
from Encoding.latent_embeddings_128 import extract_embeddings

out = extract_embeddings("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_128_L32_b4_cw6",
                        ["/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/AIIB23_128",
                         "/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/ATM22_128"],)          # one dir, or a list of dirs

enc_output = out["mu"]  # (N, num_latents) -> your clustering matrix

## sparse_256_L32_b4_cw6

In [ ]:
from Encoding.latent_embeddings_256 import extract_embeddings

out = extract_embeddings("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6",
                        ["/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/AIIB23_256^3",
                         "/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/ATM22_256^3"],)          # one dir, or a list of dirs

enc_output = out["mu"]  # (N, num_latents) -> your clustering matrix

## AE_128_L32_b2_cw6

In [ ]:
from Encoding.latent_embeddings_128_AE import extract_embeddings

out = extract_embeddings("/home/ids/gmargari-24/airway_project/Data/vae_runs/AE_128_L32_b2_cw6",
                        ["/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/AIIB23_128",
                         "/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/ATM22_128"],)          # one dir, or a list of dirs

enc_output = out["z"]  

In [ ]:
enctxt = np.savetxt("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/enc0.txt", enc_output)

In [ ]:
enc_output = np.loadtxt("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/enc0.txt")
# enc_output = enc_output.astype(int) # for autoencoder only

# dendrogram, Fig. 4

In [ ]:
plt.rcParams['lines.linewidth'] = 20
fig = plt.figure(figsize=(100, 60))
fig.patch.set_facecolor('slateblue')
plt.xlabel("Cluster Index", fontsize=250, color='white')
plt.ylabel("Euclidean Distances", fontsize=250, color='white')

# --- compute linkage once, and derive the 2-cluster line height from the data ---
Z = shc.linkage(enc_output, method='ward')
thr = (Z[-1, 2] + Z[-2, 2]) / 2          # midpoint between the last two merges = 2-cluster cut

dend = shc.dendrogram(Z,
                      truncate_mode  ='lastp',  # show only the last p merged clusters
                      p              =30,  # show only the last p merged clusters
                      leaf_rotation  =90.,
                      leaf_font_size =12.,
                      show_contracted=True,
                      # color_threshold=5500,
                      )
ax = plt.gca()
ax.tick_params(axis='x', which='major', labelsize=150, colors='white')
ax.tick_params(axis='y', which='major', labelsize=150, colors='white')

# Add a horizontal line for 2 clusters
ax.axhline(y=thr, linestyle='--', color='red', linewidth=15)

# Get the x-positions of the vertical lines
x_vals = dend['icoord']

# Iterate over the x-positions and find the intersection points with the horizontal line
for i, x in enumerate(x_vals):
    for j in range(len(x)):
        y = dend['dcoord'][i][j]
        if abs(y - thr) < 1e-5:  # check if the y-coordinate is close to the horizontal line
            ax.annotate('', xy=(x[j], y), xytext=(x[j], y + 20), arrowprops=dict(facecolor='white', edgecolor='white', arrowstyle='-|>,head_width=1,head_length=3'), fontsize=150)

plt.savefig('/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/HC.png')
plt.show()

# average intra-cluster distance, Eq. 1

In [ ]:
avg_dist = []
for c in range(1,20):
  cluster = AgglomerativeClustering(n_clusters=c, metric='euclidean', linkage='ward')
  members = cluster.fit_predict(enc_output)
  k = np.unique(members)
  num_clusters = k.shape[0]
  sum = 0
  for cluster in range(num_clusters):
    cluster_sum = 0  
    cluster_sum = pairwise_distances(enc_output[members == cluster]).mean()  
    sum += cluster_sum
  sum = sum/num_clusters
  avg_dist.append(sum)

print(avg_dist)

In [ ]:
distance_txt = np.savetxt("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/avg_dist0.txt", avg_dist)

In [ ]:
avg_dist = np.loadtxt('/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/avg_dist0.txt')
# avg_dist = avg_dist.astype(int) # for autoencoder only

In [ ]:
plt.rcParams['lines.linewidth'] = 30
fig = plt.figure(figsize=(100, 60))
fig.patch.set_facecolor('slateblue')
plt.xlabel("Number of Phenotypes", fontsize=250, color='white')
plt.ylabel("Intra-cluster Distance", fontsize=250, color='white')
plt.plot(range(1,20), avg_dist, color='mediumvioletred')
plt.xticks(range(1, 20, 1))
ax = plt.gca()
ax.set_facecolor('midnightblue')
ax.tick_params(axis='x', labelsize=200, colors='white')
ax.tick_params(axis='y', labelsize=200, colors='white')
#vlines(x=14, ymin=bottom, ymax=avg_dist[15], linewidth=4, color='r')
plt.savefig('/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/elbow0.png')
plt.show()

In [ ]:
test1 = []
for i in range(1,19):    
    test = (avg_dist[i-1] - avg_dist[i]) / avg_dist[i-1]
    test1.append(test)

print(test1)

# top 3 reductions, largest first
order = sorted(range(len(test1)), key=lambda j: test1[j], reverse=True)
top3 = order[:3]
for rank, j in enumerate(top3, 1):
    k = j + 2                      # test1[j] is the drop into k = j+2 clusters
    print(f"#{rank}: k={k}  reduction={test1[j]:.4f}")

# final clustering — they hardcoded n_clusters=2; change this to the k printed by Cell 28 if it differs)

In [ ]:
cluster = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
members = cluster.fit_predict(enc_output)

# label assignment

In [ ]:
labels = members

# save labels

In [ ]:
labelstxt = np.savetxt("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/labels.txt", labels)